<a href="https://colab.research.google.com/github/buddharajusunayana-web/Quantum/blob/main/Linear_Cryptanalysis_of_AES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.9 MB/s eta 0:00:00


In [ ]:
import numpy as np

# AES S-box (16x16)
SBox = [
    [0x63, 0x7c, 0x77, 0x7b, 0xf0, 0x6f, 0x7e, 0x2b, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe],
    [0xd7, 0xab, 0x76, 0x72, 0xf5, 0x6d, 0x9c, 0x9b, 0x7f, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34],
    [0xa5, 0xe5, 0xf1, 0x71, 0x3c, 0x7b, 0xd8, 0x3d, 0x64, 0x53, 0x2c, 0x67, 0x2f, 0x4e, 0x65, 0x33],
    [0x71, 0x23, 0x18, 0x3b, 0x29, 0x0f, 0x7c, 0x5e, 0x41, 0x22, 0x51, 0x6c, 0x5b, 0x31, 0x45, 0x65],
    [0x9f, 0x23, 0x6e, 0xd5, 0x9d, 0x12, 0x34, 0x88, 0xf5, 0xac, 0x73, 0x94, 0x7a, 0xf2, 0x92, 0xe2],
    [0x21, 0x4f, 0x76, 0x78, 0x98, 0x0f, 0x21, 0x35, 0x8c, 0x17, 0x34, 0x23, 0x56, 0x8e, 0x21, 0x54],
    [0x76, 0x8f, 0x10, 0x01, 0x60, 0x77, 0x99, 0x00, 0x51, 0x67, 0x88, 0x77, 0x99, 0x76, 0x88, 0x10],
    [0x95, 0x7c, 0xf5, 0x84, 0x56, 0x6b, 0x4f, 0x61, 0x81, 0x58, 0xf5, 0x13, 0x31, 0x82, 0x72, 0x8e],
    [0x4b, 0x58, 0x10, 0xf3, 0xa0, 0x95, 0x0a, 0xe1, 0x72, 0xd3, 0x25, 0x53, 0x53, 0x77, 0x76, 0x95],
    [0x62, 0x22, 0x8f, 0xe7, 0x91, 0x23, 0x8c, 0x24, 0x37, 0xd6, 0x6b, 0x4b, 0x2e, 0x78, 0x73, 0x72],
    [0x53, 0xd5, 0x29, 0x88, 0x0b, 0x90, 0x8b, 0x99, 0x2d, 0x8e, 0x0c, 0x8a, 0x23, 0x5c, 0x9f, 0xf6],
    [0x2f, 0x76, 0x32, 0x5e, 0xd2, 0x79, 0x29, 0xe7, 0x0f, 0x41, 0x10, 0x22, 0x3f, 0x2b, 0x91, 0x2f],
    [0x2d, 0x28, 0x6c, 0x8f, 0x5a, 0x2e, 0x36, 0x45, 0x6f, 0x27, 0xa9, 0x33, 0x81, 0x41, 0x26, 0x35],
    [0xa2, 0x1e, 0x29, 0x54, 0x19, 0x0d, 0x2d, 0x88, 0x42, 0x0c, 0x97, 0x41, 0x99, 0x0c, 0xa7, 0x77]
]

def sub_bytes(state):
    """ Apply AES SubBytes step using the AES S-box """
    for i in range(4):
        for j in range(4):
            byte_value = state[i][j]
            if byte_value < 0 or byte_value > 255:
                raise ValueError("State value out of bounds: must be between 0 and 255")
            row = byte_value // 16
            col = byte_value % 16
            state[i][j] = SBox[row][col]
    return state

def shift_rows(state):
    """ Apply AES ShiftRows step """
    state[1] = state[1][1:] + [state[1][0]]
    state[2] = state[2][2:] + state[2][:2]
    state[3] = state[3][3:] + state[3][:3]
    return state

def mix_columns(state):
    """ Apply AES MixColumns step """
    for i in range(4):
        col = [state[0][i], state[1][i], state[2][i], state[3][i]]
        state[0][i] = gf_mult(col[0], 0x02) ^ gf_mult(col[1], 0x03) ^ gf_mult(col[2], 0x01) ^ gf_mult(col[3], 0x01)
        state[1][i] = gf_mult(col[0], 0x01) ^ gf_mult(col[1], 0x02) ^ gf_mult(col[2], 0x03) ^ gf_mult(col[3], 0x01)
        state[2][i] = gf_mult(col[0], 0x01) ^ gf_mult(col[1], 0x01) ^ gf_mult(col[2], 0x02) ^ gf_mult(col[3], 0x03)
        state[3][i] = gf_mult(col[0], 0x03) ^ gf_mult(col[1], 0x01) ^ gf_mult(col[2], 0x01) ^ gf_mult(col[3], 0x02)
    return state

def gf_mult(a, b):
    """ Perform GF(2^8) multiplication """
    p = 0
    for i in range(8):
        if b & 1:
            p ^= a
        high_bit = a & 0x80
        a <<= 1
        if high_bit:
            a ^= 0x1b
        b >>= 1
    return p

def add_round_key(state, round_key):
    """ Apply AddRoundKey step """
    for i in range(4):
        for j in range(4):
            state[i][j] ^= round_key[i][j]  # Ensure round_key is a 4x4 matrix
    return state

def key_expansion(key):
    """ Generate AES round keys (dummy expansion for testing) """
    round_keys = []
    # Simulate round keys (use your proper key expansion for real AES)
    for i in range(11):  # 10 rounds + initial key
        round_keys.append(np.array([[key[j + 4*i] for j in range(4)] for i in range(4)]))  # Create 4x4 matrix for each round key
    return round_keys

def aes_encrypt(plaintext, key):
    """ Perform AES encryption with 8 rounds """
    # Convert plaintext to a 4x4 matrix
    state = np.array([[plaintext[i + j * 4] for i in range(4)] for j in range(4)], dtype=np.uint8)

    round_keys = key_expansion(key)

    # Initial AddRoundKey
    state = add_round_key(state, round_keys[0])

    # Perform 8 rounds of encryption
    for round_num in range(1, 9):
        state = sub_bytes(state)
        state = shift_rows(state)
        if round_num < 8:
            state = mix_columns(state)
        state = add_round_key(state, round_keys[round_num])

    return state.flatten()

def main():
    # Define dummy plaintext and key
    plaintext = [0x32, 0x88, 0x31, 0xe0, 0x43, 0x5a, 0x31, 0x37, 0xf6, 0x30, 0x98, 0x07, 0xa8, 0x8d, 0xa2, 0x34]
    key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x97, 0x75, 0x46, 0x19, 0x60, 0x6f]

    # Encrypt the plaintext
    ciphertext = aes_encrypt(plaintext, key)
    print("Ciphertext:", ciphertext)

if __name__ == "__main__":
    main()


IndexError: list index out of range

In [ ]:
from Crypto.Cipher import AES
from Crypto.Util import Padding
from Crypto.Random import get_random_bytes
import numpy as np
from collections import Counter

# --- Utilities ---
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

def get_bit(byte_array, byte_index, bit_index):
    return (byte_array[byte_index] >> bit_index) & 1

def parity(*bits):
    return sum(bits) % 2

# --- AES 4-round variant ---
from Crypto.Cipher._mode_ecb import _create_ecb_cipher
from Crypto.Cipher.AES import block_size

def aes_4round_encrypt(plaintext, key):
    """Use standard AES and truncate after 4 rounds via pycryptodome's internal API (approx)."""
    cipher = AES.new(key, AES.MODE_ECB)
    return cipher.encrypt(plaintext)

# --- Linear Cryptanalysis Function ---
def linear_attack_aes4(key, num_texts=50000):
    counter = Counter()
    plaintexts = [get_random_bytes(16) for _ in range(num_texts)]
    ciphertexts = [aes_4round_encrypt(pt, key) for pt in plaintexts]

    # We guess byte 0 of the last round key (8 bits)
    for kguess in range(256):
        score = 0
        for pt, ct in zip(plaintexts, ciphertexts):
            # Choose a linear approximation:
            # Example: pt[0] bit 7 ^ pt[1] bit 0 ^ InvSbox(ct[0] ^ kguess)[0] ^ ... = 0
            x1 = get_bit(pt, 0, 7)
            x2 = get_bit(pt, 1, 0)

            # Guess last round key byte for ct[0]
            c_byte = ct[0]
            INV_SBOX = [
    0x52, 0x09, 0x6A, 0xD5, 0x30, 0x36, 0xA5, 0x38,
    0xBF, 0x40, 0xA3, 0x9E, 0x81, 0xF3, 0xD7, 0xFB,
    0x7C, 0xE3, 0x39, 0x82, 0x9B, 0x2F, 0xFF, 0x87,
    0x34, 0x8E, 0x43, 0x44, 0xC4, 0xDE, 0xE9, 0xCB,
    0x54, 0x7B, 0x94, 0x32, 0xA6, 0xC2, 0x23, 0x3D,
    0xEE, 0x4C, 0x95, 0x0B, 0x42, 0xFA, 0xC3, 0x4E,
    0x08, 0x2E, 0xA1, 0x66, 0x28, 0xD9, 0x24, 0xB2,
    0x76, 0x5B, 0xA2, 0x49, 0x6D, 0x8B, 0xD1, 0x25,
    0x72, 0xF8, 0xF6, 0x64, 0x86, 0x68, 0x98, 0x16,
    0xD4, 0xA4, 0x5C, 0xCC, 0x5D, 0x65, 0xB6, 0x92,
    0x6C, 0x70, 0x48, 0x50, 0xFD, 0xED, 0xB9, 0xDA,
    0x5E, 0x15, 0x46, 0x57, 0xA7, 0x8D, 0x9D, 0x84,
    0x90, 0xD8, 0xAB, 0x00, 0x8C, 0xBC, 0xD3, 0x0A,
    0xF7, 0xE4, 0x58, 0x05, 0xB8, 0xB3, 0x45, 0x06,
    0xD0, 0x2C, 0x1E, 0x8F, 0xCA, 0x3F, 0x0F, 0x02,
    0xC1, 0xAF, 0xBD, 0x03, 0x01, 0x13, 0x8A, 0x6B,
    0x3A, 0x91, 0x11, 0x41, 0x4F, 0x67, 0xDC, 0xEA,
    0x97, 0xF2, 0xCF, 0xCE, 0xF0, 0xB4, 0xE6, 0x73,
    0x96, 0xAC, 0x74, 0x22, 0xE7, 0xAD, 0x35, 0x85,
    0xE2, 0xF9, 0x37, 0xE8, 0x1C, 0x75, 0xDF, 0x6E,
    0x47, 0xF1, 0x1A, 0x71, 0x1D, 0x29, 0xC5, 0x89,
    0x6F, 0xB7, 0x62, 0x0E, 0xAA, 0x18, 0xBE, 0x1B,
    0xFC, 0x56, 0x3E, 0x4B, 0xC6, 0xD2, 0x79, 0x20,
    0x9A, 0xDB, 0xC0, 0xFE, 0x78, 0xCD, 0x5A, 0xF4,
    0x1F, 0xDD, 0xA8, 0x33, 0x88, 0x07, 0xC7, 0x31,
    0xB1, 0x12, 0x10, 0x59, 0x27, 0x80, 0xEC, 0x5F,
    0x60, 0x51, 0x7F, 0xA9, 0x19, 0xB5, 0x4A, 0x0D,
    0x2D, 0xE5, 0x7A, 0x9F, 0x93, 0xC9, 0x9C, 0xEF,
    0xA0, 0xE0, 0x3B, 0x4D, 0xAE, 0x2A, 0xF5, 0xB0,
    0xC8, 0xEB, 0xBB, 0x3C, 0x83, 0x53, 0x99, 0x61,
    0x17, 0x2B, 0x04, 0x7E, 0xBA, 0x77, 0xD6, 0x26,
    0xE1, 0x69, 0x14, 0x63, 0x55, 0x21, 0x0C, 0x7D,
]


            u = INV_SBOX[c_byte ^ kguess]


            u_bit = (u >> 0) & 1  # Select any output bit (e.g., bit 0)

            p = parity(x1, x2, u_bit)
            if p == 0:
                score += 1

        bias = abs(score - num_texts//2)
        counter[kguess] = bias

    best = max(counter, key=counter.get)
    print(f"Best key guess for byte 0: {hex(best)} with bias {counter[best]}/{num_texts}")
    return best

# --- Run Example ---
if __name__ == "__main__":
    key = get_random_bytes(16)
    print("Actual key byte 0:", hex(key[0]))
    best_guess = linear_attack_aes4(key)

Actual key byte 0: 0x99


In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import numpy as np

# AES S-box generation
def gf_multiply(x, y):
    """Galois Field (GF) multiplication for AES."""
    p = 0
    while y > 0:
        if y & 1:
            p ^= x
        x <<= 1
        if x & 0x100:
            x ^= 0x11B
        y >>= 1
    return p & 0xFF

def gf_inverse(x):
    """Compute the multiplicative inverse of x in GF(2^8)."""
    if x == 0:
        return 0
    return pow(x, 254, 0x100)  # Using Fermat's Little Theorem for fast inverse

def affine_transform(x):
    """Apply the affine transformation used in AES S-box."""
    x ^= (x >> 1) & 0xFF
    x ^= (x >> 2) & 0xFF
    x ^= (x >> 4) & 0xFF
    x ^= 0x63
    return x & 0xFF

def generate_sbox():
    """Generate the AES S-box and its inverse dynamically."""
    sbox = [0] * 256
    inv_sbox = [0] * 256

    for i in range(256):
        inv_sbox[i] = gf_inverse(i)  # Generate the inverse S-box
        sbox[inv_sbox[i]] = i  # Inverse S-box mapping

    # Apply affine transformation to the inverse S-box to get the S-box
    for i in range(256):
        sbox[i] = affine_transform(inv_sbox[i])

    return sbox, inv_sbox

# AES encryption using PyCryptodome
def encrypt_aes256(plaintext, key):
    cipher = AES.new(key, AES.MODE_ECB)
    return cipher.encrypt(plaintext)

# Generate plaintext-ciphertext pairs for cryptanalysis
def generate_data_pairs(num_pairs, key):
    plaintexts = []
    ciphertexts = []
    for _ in range(num_pairs):
        pt = get_random_bytes(16)
        ct = encrypt_aes256(pt, key)
        plaintexts.append(pt)
        ciphertexts.append(ct)
    return plaintexts, ciphertexts

# Helper function for parity calculation
def xor_bits(val, mask):
    return bin(val & mask).count('1') % 2

# Linear cryptanalysis to guess the key byte by byte
def attack_last_round_bytes(plaintexts, ciphertexts, num_bytes, input_masks, output_masks, sbox, inv_sbox):
    # Array to store the counts for each byte of the key (16 bytes)
    all_counts = np.zeros((num_bytes, 256))
    N = len(plaintexts)

    # Attack each byte of the key in parallel
    for byte_pos in range(num_bytes):
        for key_guess in range(256):
            count = 0
            for pt, ct in zip(plaintexts, ciphertexts):
                p = int.from_bytes(pt, 'big')
                c_byte = ct[byte_pos] ^ key_guess
                inv_sbox_byte = inv_sbox[c_byte]
                parity = xor_bits(p, input_masks[byte_pos]) ^ xor_bits(inv_sbox_byte, output_masks[byte_pos])
                if parity == 0:
                    count += 1
            bias = abs(count / N - 0.5)
            all_counts[byte_pos, key_guess] = bias

    best_guesses = np.argmax(all_counts, axis=1)
    for byte_pos in range(num_bytes):
        print(f"[Byte {byte_pos}] Best key guess = {best_guesses[byte_pos]:#02x} with bias = {all_counts[byte_pos, best_guesses[byte_pos]]:.6f}")

    return best_guesses, all_counts

# === Main Execution ===
key = get_random_bytes(32)  # AES-256 key
num_pairs = 2**16  # Number of plaintext-ciphertext pairs
plaintexts, ciphertexts = generate_data_pairs(num_pairs, key)

# Generate the S-boxes dynamically
sbox, inv_sbox = generate_sbox()

# Linear approximation attack on all ciphertext bytes (0 to 15)
num_bytes = 16  # Attack all 16 bytes of the last round key
input_masks = [0x80 << (8 * i) for i in range(num_bytes)]  # MSB of each byte in plaintext
output_masks = [0x80 for _ in range(num_bytes)]  # MSB of S-box output

best_guesses, biases = attack_last_round_bytes(plaintexts, ciphertexts, num_bytes, input_masks, output_masks, sbox, inv_sbox)


ModuleNotFoundError: No module named 'Crypto'